# WhisperWatch audio-aware experiment

Do four simple audio measurements help catch high-error transcripts? The control and audio-aware models share training clips, labels, Whisper settings, and tree parameters. The audio-aware model gets four extra inputs. This experiment saves its models separately from the app.

Open this notebook in a hosted Google Colab **T4 GPU** session. Run each step in order.

## 1. Install dependencies
The decoder uses the same pinned Large V3 checkpoint and software versions as the previous AMI experiment.

In [ ]:
%pip install -q "pandas==2.2.3" "pyarrow>=23.0.1" "scikit-learn==1.8.0" "joblib==1.5.3" "faster-whisper==1.2.1" "ctranslate2==4.9.0" "av==16.0.1" "huggingface-hub==1.32.0" "nvidia-cublas-cu12" "nvidia-cudnn-cu12==9.*"

## 2. Load the frozen sample
Upload `whisperwatch-audio-aware-code.zip`. It includes metadata only. This plan samples headset clips of 2 to 25 seconds from the pinned [AMI Meeting Corpus](https://huggingface.co/datasets/edinburghcstr/ami), licensed CC BY 4.0. The prior AMI holdout is excluded from every split. Only training can reuse earlier training groups. The tuning and evaluation groups are fresh. Meeting families connected by a shared speaker are kept together. Each group contributes clips from its shard with the most eligible clips to limit transfer. The fixed seed defines the group split and draws the clips before transcriptions or WER are computed.

The holdout is previously unexamined AMI data from the same corpus. It tests generalization within AMI headset speech. It does not establish generalization to all subtitle audio or rule out Whisper pretraining exposure. The notebook prints the frozen counts before downloading 7.28 GB across 27 shards. It keeps only the sampled audio after each shard. On a T4, budget roughly 20–35 minutes of transcription plus package setup, audio and checkpoint downloads.

In [ ]:
import json
import sys
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

from google.colab import files
from IPython.display import display

project = Path("/content/whisperwatch_audio_aware")
project.mkdir(exist_ok=True)

def upload_zip(prompt):
    print(prompt)
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Choose one ZIP.")
    return BytesIO(next(iter(uploaded.values())))

with ZipFile(upload_zip("Upload whisperwatch-audio-aware-code.zip.")) as bundle:
    for member in bundle.namelist():
        parent, filename = member.split("/")
        if Path(filename).name != filename:
            raise ValueError("Unexpected upload path.")
        if (parent == "src" and filename.endswith(".py")) or member in {"plan/manifest.csv", "plan/preflight.json"}:
            path = project / member
            path.parent.mkdir(exist_ok=True)
            path.write_bytes(bundle.read(member))

sys.path.insert(0, str(project))
plan = project / "plan"
output = project / "results"
info = json.loads((plan / "preflight.json").read_text())
display(info["partitions"])
print(f'Audio transfer: {info["audio_shard_bytes"] / 1e9:.2f} GB. Whisper weights: about 3.1 GB if uncached.')


## 3. Run the comparison
Upload your original `whisperwatch-boosted-results.zip`. The runner checks its checksum. It downloads one audio shard at a time, runs Whisper once per clip, and separately times the audio feature extraction, including the extra decode. Whisper uses GPU float16, English transcription, beam size 5, VAD off, and mono audio resampled to 16 kHz.

The four inputs are overall RMS, fraction of 20 ms frames with RMS at most 0.01, fraction of decoded samples with absolute amplitude at least 0.999, and standard deviation of 20 ms frame RMS. Near-full-scale samples are a clipping proxy after resampling and mixing. None of these measurements estimates noise level or uses transcript text. If audio measurement fails for an otherwise evaluable clip, the audio model uses its built-in missing-value handling.

Both trees fit on training recordings only. The separate tuning groups are reserved as a coverage check since no settings are tuned. Human references determine the WER above 20% label. They do not enter scoring. The six queues share the same eligible evaluation clips. Each selects the highest-ranked fifth with ties broken by clip ID. If failures leave a count that is not divisible by five, review count rounds down.

The primary feature comparison is audio-aware versus control, also checked as their predefined 50/50 hybrids. Compare the new hybrid against native confidence and the original hybrid to assess practical value. The paired bootstrap uses 2,000 draws of whole linked meeting groups and recalculates the review queues each time. Inspect the leave-one-group-out table to see whether one group drives a gain. Treat one or two extra detections as exploratory. A 3–5 point recall gain with consistent direction and uncertainty supporting improvement is more persuasive.

In [ ]:
from src.audio_aware_experiment import run

original = upload_zip("Upload the original whisperwatch-boosted-results.zip.")
performance, intervals, runtime = run(plan, original, output)
display(performance)
display(intervals)
print(json.dumps({key: value for key, value in runtime.items() if key != "settings"}, indent=2))


## 4. Download the result ZIP
The ZIP includes both experimental model artifacts, all six review queues, examples gained and lost, and the extra audio processing time. Keep these results outside Git.

In [ ]:
download = Path("/content/whisperwatch-audio-aware-results.zip")
with ZipFile(download, "w", ZIP_DEFLATED) as bundle:
    for path in sorted(output.iterdir()):
        if path.suffix in {".csv", ".json", ".joblib", ".md"} and path.name != "live_input.json":
            bundle.write(path, path.name)
    bundle.write(plan / "preflight.json", "preflight.json")
files.download(str(download))
